# RQ2b results figures -- plan

Builds the figure backing RQ2b's attribution claims in
`documentation/august_draft/5_Chapter_results_evaluation/f_written_draft_v1_18thaug.tex`.
4survey only, by design -- RQ2b has no 6survey arm anywhere in this project.

**Figure in this notebook**

| Figure | Status | Question it answers |
|---|---|---|
| R2b-1 | **Must-have** | Which variables are large AND stable across model families, and does adding more environmental data close the leftover spatial gap? |

**Not built here (cut for time)**: R2b-2 (CanopyCover-dropped ablation) and R2b-3 (VIF diagnostic)
were both considered and dropped -- both add nothing beyond what is already precisely stated in
prose (e.g. EN R2 0.350->0.231 on CanopyCover removal), so building either would spend real time
on a confirmation-only appendix figure. If you want them later, the numbers already live in
`TEMP_results/TEMP_rq2_attribution_results_2026-08-11.tex`.

**Style / uncertainty convention**: see `notebooks/results_figures_style.py`. Panel A's whiskers
are fold-to-fold **sample SD** (5 folds), not a CI. Panel B (Moran's I) has no CI computed
anywhere in this project -- shown as a point value with p annotated as text. Panel C shows one
pooled value per compartment (across the 5 test folds), no per-compartment CI.

In [ ]:
# Purpose: make the models/ package (and notebooks/results_figures_style.py) importable from
# this notebook. Same convention already used across this project's other notebooks (e.g.
# notebooks/model_results/baseline_results.ipynb): walk upward until a folder containing both
# README.md and data/ is found -- that is the project root.

import sys
from pathlib import Path

notebook_directory = Path.cwd().resolve()
project_root = next(
    folder for folder in [notebook_directory, *notebook_directory.parents]
    if (folder / "README.md").exists() and (folder / "data").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("project_root:", project_root)

In [ ]:
import json

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from models.common.geo import load_compartment_boundaries, load_plot_coordinates
from models.growth_curve_attribution.residual_spatial_autocorrelation_check import compute_residual_morans_i
from notebooks.results_figures_style import COLOR_EN, COLOR_XGBOOST, COLOR_NEUTRAL_EDGE, DIVERGING_CMAP, apply_rcparams

apply_rcparams()

FIGURES_DIR = project_root / "figures" / "fig_results"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_FOLDS = 5
COHORT = "4survey"  # RQ2b has no 6survey arm
SET_RUN_NAMES = {
    "Set1": "rq2_attribution_nested_set1_baseline",
    "Set2": "rq2_attribution_nested_set2_top10",
    "Set3": "rq2_attribution_nested_set3_gated_terrain_wind_vif",
    "Set4": "rq2_attribution_nested_set4_gated_all_vif",
}

## Figure R2b-1 -- what global attribution finds, and what it still can not explain

**Research question**: RQ2b items 1-3 -- CanopyCover/thinning dominate across three converging
methods; `slope_degrees` is stable across NLME/EN while `topex` is stable only within NLME; does
the residual's own spatial clustering (Moran's I) shrink as more environmental data is added?

**Data**: Panel A -- Set4 `elastic_net_coefficients.csv` and `nlme_fixed_effects.json`, all 5
folds, `outputs/spatial_block_kfold/rq2_attribution_nested_set4_gated_all_vif/4survey/fold_{0-4}/`.
Panel B -- pooled 5-fold `predictions.csv` (`elastic_net_predicted`, `xgboost_predicted` columns)
for all four sets, joined to plot coordinates, residual Moran's I computed live via the project's
own `compute_residual_morans_i()` -- same function/weights used for every other Moran's I check in
this project. Panel C -- Set4's own pooled EN residual, aggregated to compartment mean, mapped via
`load_compartment_boundaries()`.

**Encoding**: 3-panel composite. Panel A: coefficient forest plot, one row per Set4 variable,
NLME and EN as two side-by-side points per row (not overlaid). Panel B: Moran's I by set, one line
per method (EN, XGBoost). Panel C: diverging compartment-mean residual map, centred at 0.

**Caption must state**: Panel C shows leftover unexplained error, not a spatially-varying
relationship -- unlike RQ3's GNNWR, none of RQ2b's methods (NLME, EN, XGBoost) produce a
coefficient that varies by location.

In [ ]:
# --- Panel A data: Set4 EN + NLME coefficients, mean +/- SD across 5 folds ---
set4_dir = project_root / "outputs" / "spatial_block_kfold" / SET_RUN_NAMES["Set4"] / COHORT

en_fold_frames = []
nlme_fold_frames = []
for fold in range(N_FOLDS):
    fold_dir = set4_dir / f"fold_{fold}"
    en_coef = pd.read_csv(fold_dir / "elastic_net_coefficients.csv", index_col=0)
    en_coef.columns = ["coefficient"]
    en_coef["fold"] = fold
    en_fold_frames.append(en_coef.reset_index(names="variable"))

    with open(fold_dir / "nlme_fixed_effects.json") as f:
        nlme_json = json.load(f)["fixed_effects"]
    nlme_fold = pd.DataFrame(
        [{"variable": var, "coefficient": vals["coefficient"], "fold": fold} for var, vals in nlme_json.items()]
    )
    nlme_fold_frames.append(nlme_fold)

en_all_folds = pd.concat(en_fold_frames, ignore_index=True)
nlme_all_folds = pd.concat(nlme_fold_frames, ignore_index=True)

en_summary = en_all_folds.groupby("variable")["coefficient"].agg(["mean", "std"]).rename(
    columns={"mean": "en_mean", "std": "en_sd"})
nlme_summary = nlme_all_folds.groupby("variable")["coefficient"].agg(["mean", "std"]).rename(
    columns={"mean": "nlme_mean", "std": "nlme_sd"})
panel_a_data = en_summary.join(nlme_summary, how="outer").fillna(0)
panel_a_data = panel_a_data.reindex(panel_a_data["en_mean"].abs().sort_values(ascending=True).index)
print(panel_a_data.tail(10))

In [ ]:
# --- Panel B + C data: pooled predictions per set, residual, Moran's I, and Set4's own map ---
coordinates = load_plot_coordinates()

panel_b_rows = []
set4_pooled_with_xy = None
for set_label, run_name in SET_RUN_NAMES.items():
    frames = []
    for fold in range(N_FOLDS):
        p = project_root / "outputs" / "spatial_block_kfold" / run_name / COHORT / f"fold_{fold}" / "predictions.csv"
        frames.append(pd.read_csv(p))
    pooled = pd.concat(frames, ignore_index=True)
    pooled = pooled[pooled["split"] == "test"] if "split" in pooled.columns else pooled
    pooled = pooled.merge(coordinates, on="identification", how="left")
    pooled["en_residual"] = pooled["observed"] - pooled["elastic_net_predicted"]
    pooled["xgb_residual"] = pooled["observed"] - pooled["xgboost_predicted"]

    # compute_residual_morans_i returns 5 values (semivariogram-informed distance-band
    # convention) -- range_m differs per residual field, so it is kept alongside each row rather
    # than assumed to match across methods/sets.
    en_i, en_p, en_n, en_range_m, en_status = compute_residual_morans_i(pooled["x"], pooled["y"], pooled["en_residual"])
    xgb_i, xgb_p, xgb_n, xgb_range_m, xgb_status = compute_residual_morans_i(pooled["x"], pooled["y"], pooled["xgb_residual"])
    panel_b_rows.append({"set": set_label, "method": "Elastic Net", "morans_i": en_i, "p_value": en_p, "range_m": en_range_m})
    panel_b_rows.append({"set": set_label, "method": "XGBoost", "morans_i": xgb_i, "p_value": xgb_p, "range_m": xgb_range_m})
    print(f"{set_label}: EN Moran's I={en_i:.3f} (p={en_p:.3f}, range={en_range_m:.0f}m), "
          f"XGB Moran's I={xgb_i:.3f} (p={xgb_p:.3f}, range={xgb_range_m:.0f}m)")

    if set_label == "Set4":
        set4_pooled_with_xy = pooled  # reused directly for Panel C, no re-pooling

panel_b_data = pd.DataFrame(panel_b_rows)

In [ ]:
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[1.1, 1], height_ratios=[1.6, 1])
ax_a = fig.add_subplot(gs[:, 0])
ax_b = fig.add_subplot(gs[1, 1])
ax_c = fig.add_subplot(gs[0, 1])

# --- Panel A: coefficient forest plot ---
y_positions = np.arange(len(panel_a_data))
ax_a.errorbar(panel_a_data["en_mean"], y_positions - 0.15, xerr=panel_a_data["en_sd"],
              fmt="o", color=COLOR_EN, markersize=5, capsize=2, label="Elastic Net")
ax_a.errorbar(panel_a_data["nlme_mean"], y_positions + 0.15, xerr=panel_a_data["nlme_sd"],
              fmt="s", color=COLOR_NEUTRAL_EDGE, markersize=5, capsize=2, label="NLME")
ax_a.axvline(0, color="black", linewidth=0.8)
ax_a.set_yticks(y_positions)
ax_a.set_yticklabels(panel_a_data.index, fontsize=7)
ax_a.set_xlabel("Standardised coefficient (Set4, mean +/- SD, 5 folds)")
ax_a.set_title("Panel A: coefficient comparison", fontsize=10)
ax_a.legend(fontsize=8, frameon=False, loc="lower right")

# --- Panel B: Moran's I by set ---
for method, color in [("Elastic Net", COLOR_EN), ("XGBoost", COLOR_XGBOOST)]:
    subset = panel_b_data[panel_b_data["method"] == method]
    ax_b.plot(subset["set"], subset["morans_i"], marker="o", color=color, label=method, linewidth=1.8)
ax_b.set_ylim(0, 0.8)
ax_b.set_ylabel("Residual Moran's I")
ax_b.set_title("Panel B: spatial clustering by set", fontsize=10)
ax_b.legend(fontsize=8, frameon=False)

# --- Panel C: Set4 EN residual, compartment mean, mapped ---
compartment_mean = set4_pooled_with_xy.groupby("cpmt")["en_residual"].mean().reset_index()
boundaries = load_compartment_boundaries()
mapped = boundaries.merge(compartment_mean, on="cpmt", how="left")
vmax = mapped["en_residual"].abs().max()
norm = mpl.colors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax)
mapped.plot(column="en_residual", cmap=DIVERGING_CMAP, norm=norm, ax=ax_c,
            edgecolor="white", linewidth=0.2, missing_kwds={"color": "#DDDDDD"})
sm = plt.cm.ScalarMappable(cmap=DIVERGING_CMAP, norm=norm)
fig.colorbar(sm, ax=ax_c, shrink=0.8, label="Mean EN residual (m)\nleftover error, not a local effect")
ax_c.set_title("Panel C: Set4 leftover residual (compartment mean)", fontsize=10)
ax_c.set_aspect("equal")
ax_c.set_xticks([]); ax_c.set_yticks([])

fig.suptitle("Global attribution: what it finds (A), and what it still cannot explain (B, C)", fontsize=12)
plt.tight_layout()
plt.show()